# 05 - Hyperparameter Tuning

**Objectif :** optimiser CatBoost avec Optuna sur train/validation, sans toucher au test final.

In [ ]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

## 1. Load processed development partitions

In [ ]:
from credit_risk_lab.infrastructure.data_sources import CSVDataSourceConfig, CSVDatasetRepository
from credit_risk_lab.infrastructure.modeling import CreditRiskPreprocessor

processed_train_df = CSVDatasetRepository(
    CSVDataSourceConfig(path=settings.train_path)
).load()
processed_validation_df = CSVDatasetRepository(
    CSVDataSourceConfig(path=settings.validation_path)
).load()
preprocessor = CreditRiskPreprocessor.load(settings.preprocessing_artifact_path)

{
    "processed_train": processed_train_df.shape,
    "processed_validation": processed_validation_df.shape,
    "preprocessor_features": len(preprocessor.feature_names_),
    "reserved_final_test_path": str(settings.raw_test_path),
}

## 2. Build train and validation matrices

In [ ]:
target_column = settings.target_column

x_train = processed_train_df.drop(columns=[target_column])
y_train = processed_train_df[target_column]
x_validation = processed_validation_df.drop(columns=[target_column])
y_validation = processed_validation_df[target_column]

pd.DataFrame(
    [
        {"split": "train", "rows": len(y_train), "positive_rate": y_train.mean()},
        {
            "split": "validation",
            "rows": len(y_validation),
            "positive_rate": y_validation.mean(),
        },
    ]
)

## 3. Rebuild baseline ranking

In [ ]:
from credit_risk_lab.infrastructure.modeling import (
    BestModelSelector,
    BoostingModelTrainer,
)

trainer = BoostingModelTrainer(random_state=settings.random_state)
baseline_results = trainer.fit(
    x_train,
    y_train,
    x_validation,
    y_validation,
)
baseline_metrics = trainer.results_frame(baseline_results)
best_baseline = BestModelSelector(metric=settings.selection_metric).select(
    baseline_results
)

display(baseline_metrics.round(4))
{
    "best_baseline_model": best_baseline.model_name,
    "model_selected_for_optuna": "CatBoost",
}

## 4. Baseline metric visuals

In [ ]:
from credit_risk_lab.infrastructure.visualization import plot_model_comparison

catboost_baseline_metrics = baseline_metrics[
    baseline_metrics["model"].eq("CatBoost")
].copy()

plot_model_comparison(baseline_metrics).show()
display(catboost_baseline_metrics.round(4))

## 5. Configure CatBoost Optuna search

In [ ]:
model_to_optimize = "CatBoost"
n_trials = 50
optimization_metric = "roc_auc"

if best_baseline.model_name != model_to_optimize:
    print(
        f"Baseline winner is {best_baseline.model_name}, "
        f"but this project decision is to optimize {model_to_optimize}."
    )

{
    "model_to_optimize": model_to_optimize,
    "n_trials": n_trials,
    "optimization_metric": optimization_metric,
    "test_policy": "data/raw/test.csv remains untouched",
}

## 6. Run Optuna search on CatBoost

In [ ]:
from credit_risk_lab.infrastructure.modeling import CatBoostOptunaTuner

catboost_tuner = CatBoostOptunaTuner(
    random_state=settings.random_state,
    metric=optimization_metric,
    n_trials=n_trials,
)
tuning_result = catboost_tuner.tune(
    x_train=x_train,
    y_train=y_train,
    x_validation=x_validation,
    y_validation=y_validation,
)

display(tuning_result.trials.head(10))
pd.DataFrame(
    [
        {
            "model": tuning_result.model_name,
            "best_value": tuning_result.best_value,
            "threshold": tuning_result.threshold,
            **tuning_result.metrics,
        }
    ]
).round(4)

## 7. Analyze tuning improvement

In [ ]:
from credit_risk_lab.infrastructure.visualization import (
    plot_metric_improvement,
    plot_optuna_param_importance,
    plot_optuna_parameter_slices,
    plot_optuna_trials,
)

tuned_metrics = pd.DataFrame(
    [
        {
            "model": tuning_result.model_name,
            "threshold": tuning_result.threshold,
            **tuning_result.metrics,
        }
    ]
)

plot_optuna_trials(tuning_result.trials, metric=optimization_metric).show()
plot_optuna_param_importance(
    tuning_result.study,
    metric=optimization_metric,
).show()
plot_optuna_parameter_slices(
    tuning_result.trials,
    metric=optimization_metric,
    parameters=[
        "iterations",
        "depth",
        "learning_rate",
        "l2_leaf_reg",
        "random_strength",
        "auto_class_weights",
    ],
).show()
plot_metric_improvement(
    baseline_metrics,
    tuned_metrics,
    model_name="CatBoost",
).show()

comparison = pd.concat(
    [
        catboost_baseline_metrics.assign(stage="baseline"),
        tuned_metrics.assign(stage="optuna_tuned"),
    ],
    ignore_index=True,
)
display(comparison.round(4))

## 8. Optimized CatBoost feature importance

In [ ]:
from credit_risk_lab.infrastructure.modeling import CatBoostFeatureImportanceAnalyzer
from credit_risk_lab.infrastructure.visualization import plot_feature_importance

top_n_features = 20
optimized_importance = CatBoostFeatureImportanceAnalyzer(
    tuning_result.model,
    feature_names=x_train.columns.tolist(),
).importance_frame(top_n=top_n_features)

display(optimized_importance.round(4))
plot_feature_importance(
    optimized_importance,
    title=f"Top {top_n_features} optimized CatBoost feature importances",
).show()

## 9. Save optimized CatBoost bundle

In [ ]:
from credit_risk_lab.application.workflows import current_git_commit
from credit_risk_lab.infrastructure.modeling import JoblibModelBundleRepository, sha256_file

metadata = {
    "model_name": tuning_result.model_name,
    "validation_metrics": tuning_result.metrics,
    "best_hyperparameters": tuning_result.best_parameters,
    "optimization_engine": "optuna",
    "optimization_metric": optimization_metric,
    "optuna_best_value": tuning_result.best_value,
    "optuna_trials": n_trials,
    "selection_metric": optimization_metric,
    "split_strategy": "raw_train_to_processed_train_validation_then_external_test",
    "training_dataset_sha256": sha256_file(settings.train_path),
    "validation_dataset_sha256": sha256_file(settings.validation_path),
    "preprocessing_artifact_sha256": sha256_file(settings.preprocessing_artifact_path),
    "models_config_sha256": sha256_file(settings.models_config_path),
    "git_commit": current_git_commit(),
    "target_definition": "loan_status=1 is the synthetic positive risk class",
}

bundle_path = JoblibModelBundleRepository().save(
    settings.model_bundle_path,
    model=tuning_result.model,
    preprocessor=preprocessor.transformer,
    threshold=tuning_result.threshold,
    metadata=metadata,
)

{
    "selected_tuned_model": tuning_result.model_name,
    "threshold": tuning_result.threshold,
    "parameters": tuning_result.best_parameters,
    "bundle_path": str(bundle_path),
}